# CFRBM on MovieLens 100k

This notebook installs `cfrbm` from [GitHub](https://github.com/felipecruz/CFRBM), downloads the MovieLens 100k dataset, trains the user-based model on the `u1` split with the `initial_experiment` config from `ubased.json`, and plots test MAE and RMSE for each epoch. It doesn't need a local checkout of the repository.

A full 100-epoch run takes about 13 minutes on CPU. Lower `EPOCHS` below for a quicker look.

In [ ]:
%pip install -q "cfrbm @ git+https://github.com/felipecruz/CFRBM.git@main" matplotlib

In [ ]:
import json
import os
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from cfrbm.dataset import load_dataset
from cfrbm.device import select_device
from cfrbm.user_based import run

# The dataset and the results JSON written by `run` live in this directory.
WORK_DIR = Path('cfrbm-data').resolve()
WORK_DIR.mkdir(exist_ok=True)
os.chdir(WORK_DIR)

EPOCHS = 100      # config default; each epoch is ~8 s on CPU
DEVICE = 'auto'   # auto, cpu, mps or cuda[:index]
SEED = 17

## Download the dataset

The archive and its extracted `ml-100k/` folder are saved in `cfrbm-data/`. Both steps are skipped if the files already exist.

In [ ]:
DATA_URL = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
zip_path = WORK_DIR / 'ml-100k.zip'
data_dir = WORK_DIR / 'ml-100k'

if not zip_path.exists():
    print(f'Downloading {DATA_URL} ...')
    urllib.request.urlretrieve(DATA_URL, zip_path)

if not (data_dir / 'u1.base').exists():
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(WORK_DIR)

print(sorted(p.name for p in data_dir.glob('u1.*')))

## Train

The experiment below is the `initial_experiment` config from the repository's `ubased.json`. `run` trains the model and evaluates it on the test split after every epoch. It writes the results to `<config>_<experiment>.json` in `cfrbm-data/`.

In [ ]:
experiment = {
    'name': '100k-u1xu1',
    'train_path': 'ml-100k/u1.base',
    'test_path': 'ml-100k/u1.test',
    'sep': '	',
}
config = {
    'name': 'initial_experiment',
    'epochs': EPOCHS,
    'batch_size': 10,
    'number_hidden': 100,
    'ks': [1],
    'momentums': [0.5, 0.6],
    'l_w': [0.0005],
    'l_v': [0.0005],
    'l_h': [0.0005],
    'decay': 0.0002,
}

all_users, all_movies, tests = load_dataset(
    experiment['train_path'], experiment['test_path'], experiment['sep'],
    user_based=True)
print(f'{len(all_users)} users, {len(all_movies)} movies, '
      f'{sum(map(len, tests.values()))} test ratings')

np.random.seed(SEED)  # weight initialization uses NumPy's global RNG
run(experiment['name'], experiment['train_path'], config, all_users,
    all_movies, tests, None, experiment['sep'], device=select_device(DEVICE))

## Error over epochs

In [ ]:
results_path = WORK_DIR / f"{config['name']}_{experiment['name']}.json"
results = json.loads(results_path.read_text())['results']

epochs = [r['iteration'] + 1 for r in results]
mae = [r['mae'] for r in results]
rmse = [r['rmse'] for r in results]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, rmse, label='RMSE')
ax.plot(epochs, mae, label='MAE')

# Mark where the momentum schedule steps up.
for prev, cur in zip(results, results[1:]):
    if cur['momentum'] != prev['momentum']:
        ax.axvline(cur['iteration'] + 1, color='grey', linestyle=':', linewidth=1)
        ax.text(cur['iteration'] + 1, 0.02, f" momentum {cur['momentum']}",
                transform=ax.get_xaxis_transform(), color='grey')

best = int(np.argmin(rmse))
ax.scatter([epochs[best]], [rmse[best]], zorder=3, color='C0')
ax.annotate(f'best RMSE {rmse[best]:.4f}', (epochs[best], rmse[best]),
            xytext=(0, -16), textcoords='offset points', ha='center', color='C0')

ax.set_xlabel('Epoch')
ax.set_ylabel('Test error (ratings 1–5)')
ax.set_title(f"CFRBM user-based — {experiment['name']}")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print(f'Final: MAE {mae[-1]:.4f}, RMSE {rmse[-1]:.4f} '
      f'(best RMSE {rmse[best]:.4f} at epoch {epochs[best]})')